<a href="https://colab.research.google.com/github/emilsar/Cedars/blob/main/Project3/Bayesian_Optimization_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import cho_factor, cho_solve
!pip install scikit-optimize

# ============================================================================
# STEP 1: Define the black-box function (what we're optimizing)
# ============================================================================
def step_function(x):
    """
    The 'black box' function we're trying to optimize.
    In real applications, this could be:
    - Training a neural network and returning validation accuracy
    - Running an expensive simulation
    - A physical experiment

    For demonstration, we use a continuous step function made from sigmoids.
    """
    step1 = 1.5 / (1 + np.exp(-10 * (x + 1.5)))
    step2 = 1.5 / (1 + np.exp(-10 * (x - 0.0)))
    step3 = 1.5 / (1 + np.exp(-10 * (x - 0.7)))
    step4 = -2.0 / (1 + np.exp(-10 * (x - 1.5)))
    return step1 + step2 + step3 + step4 + 0.5
print(step_function(-2.5))

0.5000680968239045


In [6]:
# ============================================================================
# STEP 2: Implement Gaussian Process
# ============================================================================
def rbf_kernel(x1, x2, length_scale=1.0):
    """RBF (Radial Basis Function) kernel: k(x,x') = exp(-||x-x'||²/2ℓ²)"""
    sqdist = np.sum((x1[:, None] - x2[None, :]) ** 2, axis=2)
    return np.exp(-0.5 * sqdist / length_scale**2)

def gp_predict(X_train, y_train, X_test, length_scale=1.0, noise=1e-8):
    """
    Compute GP posterior: μ(x) and σ(x)

    Returns:
        mu: posterior mean at X_test points
        sigma: posterior standard deviation at X_test points
    """
    # Compute covariance matrices
    K = rbf_kernel(X_train, X_train, length_scale) + noise * np.eye(len(X_train))
    K_s = rbf_kernel(X_train, X_test, length_scale)
    K_ss = rbf_kernel(X_test, X_test, length_scale)

    # Solve using Cholesky decomposition (numerically stable)
    L = cho_factor(K, lower=True)
    alpha = cho_solve(L, y_train)

    # Posterior mean: μ(x) = k*ᵀ K⁻¹ y
    mu = K_s.T @ alpha

    # Posterior variance: σ²(x) = k(x,x) - k*ᵀ K⁻¹ k*
    v = cho_solve(L, K_s)
    var = np.diag(K_ss) - np.sum(K_s * v, axis=0)

    return mu, np.sqrt(np.maximum(var, 0))

# ============================================================================
# STEP 3: Implement UCB acquisition function
# ============================================================================
def ucb(mu, sigma, kappa=2.0):
    """Upper Confidence Bound: UCB(x) = μ(x) + κ·σ(x)"""
    return mu + kappa * sigma


In [7]:
# ============================================================================
# STEP 4: Run Bayesian Optimization
# ============================================================================
# Initial observations (starting point)
X_observed = np.array([[-2.5], [-2.0]])
y_observed = np.array([step_function(x[0]) for x in X_observed])

print("Bayesian Optimization Results")
print("="*60)
print(f"Initial observations:")
print(f"  x = {X_observed[0,0]:.1f}, f(x) = {y_observed[0]:.3f}")
print(f"  x = {X_observed[1,0]:.1f}, f(x) = {y_observed[1]:.3f}")
print()

# Dense test points for computing acquisition function
X_test = np.linspace(-3, 3, 500).reshape(-1, 1)

print (X_test.shape)

# Run 4 iterations
for iteration in range(1, 5):
    print(f"Iteration {iteration}:")

    # Fit GP and compute UCB
    mu, sigma = gp_predict(X_observed, y_observed, X_test, length_scale=1.0)
    ucb_values = ucb(mu, sigma, kappa=2.0)

    # Find next point to sample (maximize UCB)
    next_idx = np.argmax(ucb_values)
    next_x = X_test[next_idx, 0]

    print(f"  Next point: x = {next_x:.3f}")
    print(f"  UCB({next_x:.3f}) = {ucb_values[next_idx]:.3f}")

    # Evaluate the black-box function
    next_y = step_function(next_x)
    print(f"  f({next_x:.3f}) = {next_y:.3f}")

    # Update observations
    X_observed = np.vstack([X_observed, [[next_x]]])
    y_observed = np.append(y_observed, next_y)

    # Show current best
    best_idx = np.argmax(y_observed)
    print(f"  Current best: x = {X_observed[best_idx,0]:.3f}, "
          f"f(x) = {y_observed[best_idx]:.3f}")
    print()

print("="*60)
print("Optimization complete!")
print(f"Found optimum: x* = {X_observed[best_idx,0]:.3f}, "
      f"f(x*) = {y_observed[best_idx]:.3f}")
print(f"Total function evaluations: {len(X_observed)}")

Bayesian Optimization Results
Initial observations:
  x = -2.5, f(x) = 0.500
  x = -2.0, f(x) = 0.510

(500, 1)
Iteration 1:
  Next point: x = 0.246
  UCB(0.246) = 2.013
  f(0.246) = 3.398
  Current best: x = 0.246, f(x) = 3.398

Iteration 2:
  Next point: x = 0.800
  UCB(0.800) = 3.947
  f(0.800) = 4.593
  Current best: x = 0.800, f(x) = 4.593

Iteration 3:
  Next point: x = 1.293
  UCB(1.293) = 5.052
  f(1.293) = 4.773
  Current best: x = 1.293, f(x) = 4.773

Iteration 4:
  Next point: x = 1.112
  UCB(1.112) = 4.866
  f(1.112) = 4.935
  Current best: x = 1.112, f(x) = 4.935

Optimization complete!
Found optimum: x* = 1.112, f(x*) = 4.935
Total function evaluations: 6


In [18]:
# You may need: !pip install scikit-optimize
from skopt import gp_minimize
from skopt.space import Real
import numpy as np

# Define the step function (same as before)
def step_function(x):
    step1 = 1.5 / (1 + np.exp(-10 * (x + 1.5)))
    step2 = 1.5 / (1 + np.exp(-10 * (x - 0.0)))
    step3 = 1.5 / (1 + np.exp(-10 * (x - 0.7)))
    step4 = -2.0 / (1 + np.exp(-10 * (x - 1.5)))
    return step1 + step2 + step3 + step4 + 0.5

# Note: gp_minimize MINIMIZES, so we negate for maximization
def negative_step_function(x):
    return -step_function(x[0])  # x is a list with one element

# Define the search space
space = [Real(-3.0, 3.0, name='x')]

# Run Bayesian optimization with settings matching our from-scratch version
result = gp_minimize(
    negative_step_function,
    space,
    x0=[[-2.5], [-2.0]],
    n_calls=4,
    n_initial_points=0,
    acq_func='LCB',              # ← Lower Confidence Bound (negative UCB)
    kappa=2.0,                   # ← κ = 2.0 (same as our UCB implementation)
    acq_optimizer='sampling',    # ← Evaluate at many points (like our 500-point grid)
    n_points=500,                # ← Number of points to sample (matches our approach)
    random_state=42,
    verbose=True
)

print(f"\nOptimal x: {result.x[0]:.3f}")
print(f"Optimal f(x): {-result.fun:.3f}")  # Negate back for maximum
print(f"Total evaluations: {len(result.func_vals)}")

Iteration No: 1 started. Evaluating function at provided point.
Iteration No: 1 ended. Evaluation done at provided point.
Time taken: 0.2273
Function value obtained: -0.5100
Current minimum: -0.5100
Iteration No: 2 started. Evaluating function at provided point.
Iteration No: 2 ended. Evaluation done at provided point.
Time taken: 0.5891
Function value obtained: -0.5098
Current minimum: -0.5100
Iteration No: 3 started. Searching for the next optimal point.
Iteration No: 3 ended. Search finished for the next optimal point.
Time taken: 0.4719
Function value obtained: -0.5569
Current minimum: -0.5569
Iteration No: 4 started. Searching for the next optimal point.

Optimal x: -1.823
Optimal f(x): 0.557
Total evaluations: 4
